## 0. Importer les videos se trouvant sur la chaine youtube d'un artiste beninois.

In [4]:
import csv
from yt_dlp import YoutubeDL
import logging

In [5]:
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

In [6]:
URL = "https://www.youtube.com/@leroialekpehanhou/videos"

with YoutubeDL({"extract_flat": True, "quiet": True}) as ydl:
    info = ydl.extract_info(URL, download=False)

def format_temps(d):
    if not d:
        return ""
    d = int(d)
    return f"{d // 3600:02d}:{d % 3600 // 60:02d}:{d % 60:02d}"

with open("videos_alekpehanhou.csv", "w", newline="", encoding="utf-8-sig") as f:
    w = csv.writer(f, delimiter=";")
    w.writerow(["titre", "lien", "temps"])
    for e in info["entries"]:
        w.writerow([
            e.get("title", ""),
            f"https://www.youtube.com/watch?v={e['id']}",
            format_temps(e.get("duration")),
        ])

logging.info(f"{len(info['entries'])} vidéos enregistrées dans videos_alekpehanhou.csv")

2026-11-15 14:32:33,441 - INFO - 45 vidéos enregistrées dans videos_alekpehanhou.csv


## Acquisition des données
- Téléchargement des videos listées depuis la page youtube
- Extraction de l'audio en WAV
- Creation d'un CSV : titre ; lien ; temps ; chemin_video ; chemin_audio ; statut

Prérequis (Mac) :
-    brew install ffmpeg
-    pip install yt-dlp

In [7]:
import shutil
import subprocess
import sys
from pathlib import Path

In [18]:
CHAINE = "https://www.youtube.com/@leroialekpehanhou/videos"
DOSSIER = Path("../data")
VIDEO_DIR = DOSSIER / "raw" / "video"
AUDIO_DIR = DOSSIER / "raw" / "audio"
CSV_SORTIE = DOSSIER / "metadata" / "videos_alekpehanhou.csv"
MAX_VIDEOS = None        # None pour tout télécharger (tester d'abord avec un nombre entier n)
SEPARATEUR = ";"

In [9]:
def lister_videos():
    with YoutubeDL({"extract_flat": True, "quiet": True}) as ydl:
        info = ydl.extract_info(CHAINE, download=False)
    entrees = [e for e in info.get("entries", []) if e and e.get("id")]
    return entrees[:MAX_VIDEOS] if MAX_VIDEOS else entrees

In [10]:
def telecharger_video(video_id):
    cible = VIDEO_DIR / f"{video_id}.mp4"
    if cible.exists():
        return cible
    options = {
        "format": "bv*[ext=mp4]+ba[ext=m4a]/b[ext=mp4]/b",
        "merge_output_format": "mp4",
        "outtmpl": str(VIDEO_DIR / "%(id)s.%(ext)s"),
        "quiet": True,
        "no_warnings": True,
    }
    with YoutubeDL(options) as ydl:
        ydl.download([f"https://www.youtube.com/watch?v={video_id}"])
    if not cible.exists():
        raise FileNotFoundError(f"Vidéo introuvable après téléchargement : {cible}")
    return cible

In [11]:
def extraire_audio(video, video_id):
    cible = AUDIO_DIR / f"{video_id}.wav"
    if cible.exists():
        return cible
    subprocess.run(
        ["ffmpeg", "-y", "-loglevel", "error", "-i", str(video),
         "-vn", "-ac", "2", "-ar", "44100", "-c:a", "pcm_s16le", str(cible)],
        check=True,
    )
    return cible

In [16]:
def main():
    if shutil.which("ffmpeg") is None:
        sys.exit("ffmpeg est introuvable. Installe-le avec : brew install ffmpeg(macOS) ou sudo apt install ffmpeg (Linux)")

    VIDEO_DIR.mkdir(parents=True, exist_ok=True)
    AUDIO_DIR.mkdir(parents=True, exist_ok=True)

    logging.info("Récupération de la liste des vidéos...")
    videos = lister_videos()
    logging.info(f"{len(videos)} vidéo(s) à traiter.\n")

    with open(CSV_SORTIE, "w", newline="", encoding="utf-8-sig") as f:
        w = csv.writer(f, delimiter=SEPARATEUR)
        w.writerow(["titre", "lien", "temps", "chemin_video", "chemin_audio", "statut"])

        for i, e in enumerate(videos, 1):
            vid = e["id"]
            titre = e.get("title", "")
            lien = f"https://www.youtube.com/watch?v={vid}"
            temps = format_temps(e.get("duration"))
            logging.info(f"[{i}/{len(videos)}] {titre}")

            chemin_video = chemin_audio = ""
            try:
                video = telecharger_video(vid)
                chemin_video = str(video)
                audio = extraire_audio(video, vid)
                chemin_audio = str(audio)
                statut = "ok"
            except Exception as err:
                statut = f"erreur : {err}"
                logging.info(f"    -> {statut}")

            w.writerow([titre, lien, temps, chemin_video, chemin_audio, statut])
            f.flush()

    logging.info(f"\nTerminé. CSV : {CSV_SORTIE}")


In [19]:
main()

2026-11-15 14:39:39,398 - INFO - Récupération de la liste des vidéos...
2026-11-15 14:39:41,182 - INFO - 45 vidéo(s) à traiter.

2026-11-15 14:39:41,183 - INFO - [1/45] Kpo Dè Médé Nu Na Fiô
2026-11-15 14:39:41,184 - INFO - [2/45] Médé Kun No Nywè Nui Hu Se O
2026-11-15 14:39:41,185 - INFO - [3/45] Axwawlé
2026-11-15 14:39:41,185 - INFO - [4/45] Dodonusi No Gb'a Su A ?
2026-11-15 14:39:41,186 - INFO - [5/45] Jonion d'Alakénu
2026-11-15 14:39:41,186 - INFO - [6/45] Gban We Do Gbangbé
2026-11-15 14:39:41,187 - INFO - [7/45] E Non Vè Loo !
2026-11-15 14:39:41,187 - INFO - [8/45] Vodounsi Gansé Gansé Kou (Hwèdè)
2026-11-15 14:39:41,188 - INFO - [9/45] Ma Ho Houédè Nou Mé Vivè Lè (Hwèdè)
2026-11-15 14:39:41,188 - INFO - [10/45] Jéjé Yi Koo ! (Hwèdè)
2026-11-15 14:39:41,189 - INFO - [11/45] Fitè Mi Non Bo Dô E ?
2026-11-15 14:39:41,189 - INFO - [12/45] Noudé Wé W'ado B'adôwéhè Gbèdè Kô
2026-11-15 14:39:41,190 - INFO - [13/45] Gbonon Wunmè W'agbadé Non Mia Go Do
2026-11-15 14:39:41,190 - INFO

## 1. Vision de la V0
L'objectif est de construire un premier moteur capable de transformer des enregistrements musicaux en représentations exploitables, puis de produire de nouvelles séquences musicales à partir de structures rythmiques existantes.

Il faut bien distinguer deux éléments :
- La donnée audio : le fichier sonore original, qui contient la performance musicale.
- La donnée symbolique : une représentation structurée de la musique sous forme d'événements, de notes, de positions temporelles et de durées.

![Architecture](../img/architecture.png)